In [4]:
import pandas as pd
import numpy as np

# 1. Load raw data
df = pd.read_csv('Hospital_Industry_Grade_100K.csv')

print("======= DATA CLEANING PIPELINE STARTED =======")

# 2. Re-indexing & Concatenation Simulation
df['Date'] = pd.to_datetime(df['Date'])
df_part1 = df[df['Date'].dt.year <= 2023]
df_part2 = df[df['Date'].dt.year > 2023]
df_cleaned = pd.concat([df_part1, df_part2], axis=0).reset_index(drop=True)

# 3. Handling Missing Values (Imputation)
# Category column: Fill with 'Unknown'
missing_city_before = df_cleaned['City'].isna().sum()
df_cleaned['City'] = df_cleaned['City'].fillna('Unknown')

# Price column: Impute with median price of that specific medical category
missing_price_before = df_cleaned['Price'].isna().sum()
df_cleaned['Price'] = df_cleaned.groupby('Category')['Price'].transform(lambda x: x.fillna(x.median()))

# Discount column: Impute with 0 (assuming missing means no discount applied)
missing_discount_before = df_cleaned['Discount'].isna().sum()
df_cleaned['Discount'] = df_cleaned['Discount'].fillna(0)

df_cleaned['Comments'] = df_cleaned['Comments'].str.strip().str.lower()

df_cleaned = df_cleaned[(df_cleaned['Age'] <= 100) & (df_cleaned['Rating'] <= 5)]

# 4. Deduplication
duplicates_count = df_cleaned.duplicated().sum()
if duplicates_count > 0:
    df_cleaned = df_cleaned.drop_duplicates()

# 5. Export cleaned file
df_cleaned.to_csv('Hospital_Cleaned_Final.csv', index=False)

print("\n======= PIPELINE SUCCESSFUL =======")
print(f"Total Records Processed: {len(df_cleaned)}")
print(f"Missing Cities Resolved: {missing_city_before}")
print(f"Missing Prices Imputed: {missing_price_before}")
print(f"Missing Discounts Handled: {missing_discount_before}")
print(f"Duplicate Rows Removed: {duplicates_count}")

======= DATA CLEANING PIPELINE STARTED =======

======= PIPELINE SUCCESSFUL =======
Total Records Processed: 96032
Missing Cities Resolved: 3035
Missing Prices Imputed: 3040
Missing Discounts Handled: 1513
Duplicate Rows Removed: 972


In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv('Hospital_Cleaned_Final.csv')
df['Date'] = pd.to_datetime(df['Date'])
df['Year'] = df['Date'].dt.year

print("======= PHASE 2: EXPLORATORY DATA ANALYSIS =======")

# 1. Demographic Analysis: Age and Gender Insights
print("\n[1] Patient Demographics Summary:")
age_groups = pd.cut(df['Age'], bins=[0, 18, 35, 50, 65, 100, 160], 
                    labels=['0-18', '19-35', '36-50', '51-65', '66-100', '100+ Outliers'])
print(df.groupby(age_groups, observed=False)['Revenue'].agg(['count', 'sum', 'mean']))

# 2. Financial Analysis: Revenue & Profitability by Medical Category
print("\n[2] Financial Performance by Medical Category:")
category_perf = df.groupby('Category')[['Revenue', 'Profit', 'Quantity']].sum()
category_perf['Profit_Margin_%'] = (category_perf['Profit'] / category_perf['Revenue']) * 100
print(category_perf)

# 3. Channel Operational Efficiency: How patients interact with the hospital
print("\n[3] Revenue and Ratings by Communication Channel:")
channel_perf = df.groupby('Channel').agg(
    Total_Revenue=('Revenue', 'sum'),
    Average_Rating=('Rating', 'mean'),
    Total_Patients=('Customer_ID', 'count')
)
print(channel_perf)

# 4. Regional Performance: Which regions generate the highest financial value?
print("\n[4] Regional Financial Contributions:")
region_perf = df.groupby('Region')[['Revenue', 'Profit']].sum().sort_values(by='Revenue', ascending=False)
print(region_perf)

# Save these summaries as individual CSVs so you have the numbers ready for your PPT slides!
category_perf.to_csv('EDA_Category_Performance.csv')
channel_perf.to_csv('EDA_Channel_Performance.csv')
region_perf.to_csv('EDA_Region_Performance.csv')
print("\nEDA Summary spreadsheets successfully saved!")

======= PHASE 2: EXPLORATORY DATA ANALYSIS =======

[1] Patient Demographics Summary:
               count           sum          mean
Age                                             
0-18            1837  1.258828e+08  68526.281613
19-35          32296  2.404411e+09  74449.185532
36-50          28094  2.106054e+09  74964.550828
51-65          28255  2.020160e+09  71497.434161
66-100          7518  5.473781e+08  72809.004278
100+ Outliers   2000  1.586963e+08  79348.155046

[2] Financial Performance by Medical Category:
               Revenue        Profit  Quantity  Profit_Margin_%
Category                                                       
A         1.736338e+09  6.097147e+08    185334        35.114975
B         1.879961e+09  6.144827e+08    186840        32.685928
C         1.883459e+09  6.062185e+08    183949        32.186445
D         1.862824e+09  6.102388e+08    185422        32.758798

[3] Revenue and Ratings by Communication Channel:
         Total_Revenue  Average_Rating 

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set professional visualization styles
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

print("=== STARTING MASTER CLEANING & VISUALIZATION PIPELINE ===")

# 1. LOAD RAW DATA
df = pd.read_csv('Hospital_Industry_Grade_100K.csv')

# 2. SIMULATE ENTERPRISE CONCATENATION (Internship Requirement)
df['Date'] = pd.to_datetime(df['Date'])
df_part1 = df[df['Date'].dt.year <= 2023]
df_part2 = df[df['Date'].dt.year > 2023]
df_master = pd.concat([df_part1, df_part2], axis=0).reset_index(drop=True)

# 3. DROP ANOMALIES & OUTLIERS (Age > 100 & Rating > 5)
# Stripping down to biologically and logically realistic bounds
df_clean = df_master[(df_master['Age'] <= 100) & (df_master['Rating'] <= 5)].copy()

# 4. HANDLE MISSING VALUES & TEXT STANDARDIZATION
df_clean['City'] = df_clean['City'].fillna('Unknown')
df_clean['Price'] = df_clean.groupby('Category')['Price'].transform(lambda x: x.fillna(x.median()))
df_clean['Discount'] = df_clean['Discount'].fillna(0)
df_clean['Comments'] = df_clean['Comments'].str.strip().str.lower()

# Save final clean dataset
df_clean.to_csv('Hospital_Final_Cleaned_Data.csv', index=False)
print(f"Pipeline complete! Dropped {len(df) - len(df_clean)} total anomalous/outlier rows.")

# =========================================================
# PHASE 3: VISUALIZATIONS (GENERATE PLOTS FROM CLEAN DATA)
# =========================================================
print("\n=== GENERATING CLEAN VISUALIZATIONS ===")

# PLOT 1: Financial Performance by Category
cat_perf = df_clean.groupby('Category')[['Revenue', 'Profit']].sum().reset_index()
melted_cat = pd.melt(cat_perf, id_vars='Category', value_vars=['Revenue', 'Profit'], var_name='Metric', value_name='Amount')
plt.figure(figsize=(10, 6))
sns.barplot(data=melted_cat, x='Category', y='Amount', hue='Metric', palette='viridis')
plt.title('Total Revenue vs. Net Profit by Medical Category (Cleaned)', fontsize=13, fontweight='bold', pad=15)
plt.ylabel('Financial Amount (in Billions)')
plt.tight_layout()
plt.savefig('visual_1_category_financials.png', dpi=300)
plt.close()

# PLOT 2: Regional Market Share (Donut Chart)
region_data = df_clean.groupby('Region')['Revenue'].sum()
plt.figure(figsize=(6, 6))
plt.pie(region_data, labels=region_data.index, autopct='%1.1f%%', startangle=90, 
        colors=sns.color_palette('pastel')[0:4], wedgeprops=dict(width=0.4, edgecolor='w'))
plt.title('Hospital Revenue Market Share by Region', fontsize=13, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('visual_2_regional_share.png', dpi=300)
plt.close()

# PLOT 3: Patient Sentiment by Channel (Stacked Bar)
channel_sentiment = df_clean.groupby(['Channel', 'Comments']).size().unstack(fill_value=0)
channel_sentiment_pct = channel_sentiment.div(channel_sentiment.sum(axis=1), axis=0) * 100
channel_sentiment_pct.plot(kind='bar', stacked=True, color=['#fef08a', '#bbf7d0', '#fecdd3'], edgecolor='gray')
plt.title('Patient Sentiment Distribution by Engagement Channel', fontsize=13, fontweight='bold', pad=15)
plt.xlabel('Booking Channel')
plt.ylabel('Percentage (%)')
plt.xticks(rotation=0)
plt.legend(title='Feedback', labels=['Average', 'Good', 'Poor'], loc='lower right')
plt.tight_layout()
plt.savefig('visual_3_channel_sentiment.png', dpi=300)
plt.close()

# PLOT 4: Cleaned Patient Age Distribution (Histogram)
plt.figure(figsize=(10, 6))
sns.histplot(data=df_clean, x='Age', bins=20, kde=True, color='teal', edgecolor='black')
plt.title('Patient Age Demographics (Cleaned Profile: Ages 18-100)', fontsize=13, fontweight='bold', pad=15)
plt.xlabel('Patient Age')
plt.ylabel('Patient Record Count')
plt.tight_layout()
plt.savefig('visual_4_age_distribution.png', dpi=300)
plt.close()

print("All charts exported successfully! ")


=== STARTING MASTER CLEANING & VISUALIZATION PIPELINE ===
Pipeline complete! Dropped 3996 total anomalous/outlier rows.

=== GENERATING CLEAN VISUALIZATIONS ===
All charts exported successfully! 
